<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-architect-program/blob/main/Day_18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weekend 9 Lab — Build a 3-Agent Team: Researcher → Analyst → Writer

**What you'll build:** three small, specialised AI agents that work **one after another** — like three coworkers passing a task down the line:

```
Researcher  →  Analyst  →  Writer
(finds facts)  (weighs risks)  (writes the recommendation)
```

Each agent is just **one Claude call with its own job**. You will write real code, call the real Claude API, and see real output at every step — no pretend/offline code anywhere in this lab.

---

### Who this is for
AI engineers, AI architects, and career-switchers (including Oracle DBAs moving into AI) who finished the Weekend 9 theory session on multi-agent systems. No prior coding-agent experience needed. Every step is explained in plain words.

### What you'll be able to do after this lab
- **Explain**, in one sentence, what a "3-agent pipeline" actually is.
- **Build** three specialised agents, each with its own job and its own instructions.
- **Chain** them so one agent's answer becomes the next agent's input.
- **Run** the whole pipeline end to end on a real business question.
- **Explain** how this maps to the theory (roles, isolation, handoffs) and how it's different from the newest kind of multi-agent system: the **agent swarm** (Kimi K2.6 and similar).

### How this notebook is organised
| Part | What's in it |
|---|---|
| Setup | Install the SDK, connect your API key |
| One agent at a time | Build Researcher, then Analyst, then Writer — run each alone first |
| Put it together | Chain all three into one pipeline |
| Architecture | A simple picture of the whole system |
| What's next in multi-agent AI | Kimi's agent swarm + other 2026 developments |
| Mini project | Build your own version end to end |
| Revision suite | Summary, cheat sheet, assignment, quiz |


# Part 0 — What is a "3-agent pipeline"? (Recap in plain words)

Forget the words "AI agent" for a second. Think of **three coworkers** who each do one job, in order:

1. **Researcher** — reads the question and writes down the facts. Nothing else.
2. **Analyst** — reads the Researcher's facts and decides what's risky and what's promising.
3. **Writer** — reads the Analyst's notes and writes the final recommendation.

Each coworker only sees what the **previous** coworker handed them. That's it. That is the *entire* idea behind a "multi-agent pipeline."

### Why not just ask Claude to do all three jobs in one prompt?
You could. But splitting the job into three specialists has two real benefits:
- **Each prompt is shorter and more focused**, so each agent is less likely to get confused about its job.
- **You can inspect every hand-off.** If the final answer is wrong, you can look at exactly which step introduced the problem.

### The three words you need (and only these three)
| Word | Plain meaning |
|---|---|
| **Agent** | One Claude call with one job and its own instructions (its "system prompt") |
| **Pipeline** | Running the agents in a fixed order: Researcher → Analyst → Writer |
| **Hand-off** | Taking one agent's text output and pasting it into the next agent's input |

That's the whole vocabulary for this lab. Everything below is just code that does these three things.


# Part 1 — Setup (run this once)

Two cells. Run them in order, top to bottom, before anything else in this notebook.


In [1]:
# Cell 1 — install the Claude SDK
!pip install anthropic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 12.2 MB/s eta 0:00:00


In [2]:
# Cell 2 — connect your API key and create the client
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("MY_API_KEY")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"   # fast and cheap - perfect for this lab

print("Ready. Model:", MODEL)


Ready. Model: claude-haiku-4-5-20251001


> **Not using Colab?** Only Cell 2 changes: set `os.environ["ANTHROPIC_API_KEY"]` from your own environment variable or `.env` file instead of a Colab Secret. Never type a real key directly into a shared notebook.


# Part 2 — Build one agent at a time

Here is the one pattern you will repeat three times:

```python
reply = client.messages.create(
    model=MODEL,
    max_tokens=...,
    system="<this agent's job, in plain words>",
    messages=[{"role": "user", "content": "<what this agent needs to read>"}],
)
answer = reply.content[0].text
```

The **only** thing that changes between agents is the `system` instruction and what you hand it as input. Same pattern, three times. Let's build it.


## Step 1 — The Researcher
**Job:** read a business question, return facts only. No opinions, no recommendation.


In [3]:
# The Researcher: facts only, exactly 5 bullet points
RESEARCHER_SYSTEM = """You are a careful market researcher.
When given a business question, output EXACTLY 5 short bullet points of facts and
relevant numbers. No opinions. No recommendation. Just facts a decision-maker would need."""

def run_researcher(question):
    reply = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=RESEARCHER_SYSTEM,
        messages=[{"role": "user", "content": question}],
    )
    return reply.content[0].text

# Try it on its own first
question = "Should Zepto expand its 10-minute grocery delivery into Indian tier-2 cities?"
research_notes = run_researcher(question)
print(research_notes)


# Market Facts on Zepto Tier-2 Expansion

• **Current market concentration**: Zepto operates primarily in tier-1 cities (Mumbai, Delhi, Bangalore, Hyderabad); tier-2 cities represent ~35% of India's urban population but remain largely underserved by quick-commerce players

• **Unit economics challenge**: Quick-commerce 10-minute delivery requires dense order concentration; tier-2 cities average 40-50% lower population density than tier-1, increasing delivery cost per order by an estimated 25-35%

• **Competitor activity**: Blinkit and Instamart have entered select tier-2 markets (Jaipur, Pune, Lucknow); limited profitability data available, but both maintain tier-1 as primary focus

• **Consumer spending capacity**: Tier-2 urban households have ~60% lower average grocery spending than tier-1; quick-commerce basket sizes typically 25-30% smaller, reducing revenue per delivery

• **Infrastructure dependency**: Tier-2 cities have 15-20% fewer warehousing options and 30% longer last-mile d

**Expected result:** exactly 5 short, factual bullet points — no verdict, no opinion. That is correct. The Researcher's job stops here on purpose.

### ✅ Quick Check
> **Q:** Why does the Researcher's instructions say "no opinions, no recommendation"?
> **A:** Because that's the Analyst's and Writer's job. Keeping the Researcher narrow means it can't accidentally sneak a conclusion in before the risks have even been weighed.


## Step 2 — The Analyst
**Job:** read the Researcher's facts, sort them into risks and opportunities. Nothing invented.


In [4]:
# The Analyst: takes the Researcher's notes, weighs risk vs opportunity
ANALYST_SYSTEM = """You are a sharp business analyst.
You will be given research notes. Using ONLY those notes, produce two short sections:
1. "Risks (ranked High/Medium/Low)"
2. "Opportunities"
Do not invent facts that are not in the notes."""

def run_analyst(research_notes):
    reply = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=ANALYST_SYSTEM,
        messages=[{"role": "user", "content": research_notes}],
    )
    return reply.content[0].text

# Try it on its own, feeding in the Researcher's output from Step 1
analysis = run_analyst(research_notes)
print(analysis)


# Risks (ranked High/Medium/Low)

**HIGH**
- **Unit economics deterioration**: Lower population density (40-50% in tier-2) combined with 25-35% higher delivery cost per order and 25-30% smaller basket sizes creates a challenging path to profitability in tier-2 markets

**HIGH**
- **Infrastructure constraints**: 15-20% fewer warehousing options and 30% longer last-mile distances in tier-2 cities will increase operational setup costs by 40-50% per location, pressuring margins and capital deployment efficiency

**MEDIUM**
- **Reduced revenue per customer**: Tier-2 households spend ~60% less on groceries than tier-1, limiting volume growth potential and customer lifetime value relative to expansion investment

**MEDIUM**
- **Competitive saturation risk**: Blinkit and Instamart are already present in select tier-2 markets (Jaipur, Pune, Lucknow), indicating competitive awareness despite their tier-1 focus

---

# Opportunities

- **Large underserved market access**: Tier-2 cities represent 

**Expected result:** a short "Risks" list and a short "Opportunities" list, built only from the bullet points above. Nothing new was invented.

### ✅ Quick Check
> **Q:** The Analyst never saw the original question — only the Researcher's bullet points. Is that a bug?
> **A:** No — that's the **hand-off** working correctly. The Analyst's whole job is to reason over the facts it was given, not to re-answer the original question.


## Step 3 — The Writer
**Job:** read the Analyst's risks/opportunities, write one final recommendation with a clear Yes/No verdict.


In [5]:
# The Writer: turns the analysis into one tight recommendation + verdict
WRITER_SYSTEM = """You are an executive business writer.
You will be given a risk/opportunity analysis. Write a final recommendation for a CEO:
- ONE tight paragraph, plain English.
- End with exactly one line: "Verdict: Yes" or "Verdict: No".
- Use only what the analysis says. Do not add new facts."""

def run_writer(analysis):
    reply = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=WRITER_SYSTEM,
        messages=[{"role": "user", "content": analysis}],
    )
    return reply.content[0].text

# Try it on its own, feeding in the Analyst's output from Step 2
draft = run_writer(analysis)
print(draft)


Expanding into tier-2 cities presents a structurally unviable business case despite the market size opportunity. The combination of 40-50% lower population density, 25-35% higher delivery costs per order, 25-30% smaller basket sizes, and 60% lower customer spending creates unit economics that cannot support profitability—compounded by 40-50% higher per-location setup costs due to infrastructure constraints and 30% longer last-mile distances. While tier-2 cities represent 35% of India's urban population with limited quick-commerce penetration, the cost structure fundamentally breaks the unit economics model that works in tier-1 markets. Competitors like Blinkit and Instamart have already proven selective market entry, suggesting they too recognize profitability constraints rather than seeing this as an untapped goldmine. First-mover advantages in select cities cannot offset the mathematical reality that lower spending volumes, higher operational costs, and smaller baskets create a value

**Expected result:** one short paragraph ending in "Verdict: Yes" or "Verdict: No".

### ✅ Quick Check
> **Q:** Could the Writer just answer the original question directly, skipping the Researcher and Analyst?
> **A:** Yes, technically. But then one single prompt would have to gather facts, weigh risk, *and* write the recommendation — three jobs at once. Splitting it up is what keeps each step focused and easy to check.


# Part 3 — Put it together: one pipeline function

You already ran all three agents by hand above, pasting one output into the next by eye. Now let's do that automatically with one function.


In [6]:
# The full pipeline: Researcher -> Analyst -> Writer, one hand-off at a time
def run_pipeline(question):
    print("STEP 1 - Researcher is gathering facts...")
    research_notes = run_researcher(question)

    print("STEP 2 - Analyst is weighing risks and opportunities...")
    analysis = run_analyst(research_notes)

    print("STEP 3 - Writer is drafting the recommendation...")
    final_draft = run_writer(analysis)

    return {
        "research_notes": research_notes,
        "analysis": analysis,
        "final_draft": final_draft,
    }

result = run_pipeline("Should Blinkit open dark stores in Coimbatore?")

print("\n--- RESEARCH NOTES ---\n", result["research_notes"])
print("\n--- ANALYSIS ---\n", result["analysis"])
print("\n--- FINAL RECOMMENDATION ---\n", result["final_draft"])


STEP 1 - Researcher is gathering facts...
STEP 2 - Analyst is weighing risks and opportunities...
STEP 3 - Writer is drafting the recommendation...

--- RESEARCH NOTES ---
 # Coimbatore Market Facts for Dark Store Expansion

• **Market Size**: Coimbatore has ~2.2 million population with growing urban consumption; Tier-2 city with emerging quick-commerce demand but lower penetration than Metro cities

• **Current Competition**: Zepto, Dunzo, and local players already operate in Coimbatore; Blinkit's current presence is limited compared to major metros

• **Logistics Advantage**: Coimbatore's compact city geography (vs sprawling metros) could support efficient 10-15 minute delivery from fewer dark stores; proximity to Highway 48 provides supply chain benefits

• **Consumer Behavior Data**: Quick-commerce adoption in Tier-2 Indian cities growing 40-50% YoY, but average order values 15-20% lower than metros; profitability timelines longer

• **Unit Economics**: Dark store capex per locatio

**Expected result:** three clearly labelled blocks printed in order — research notes, then analysis, then the final Yes/No recommendation. Each block is built entirely from the one before it.

### Try this
Change the `question` to something you actually care about (a product decision, a hiring question, a "should we build X" question at your own company) and re-run the cell. Watch how a vague question produces vague research, which produces a vague final recommendation — the pipeline is only as good as the facts it starts with.

### ✅ Quick Check — the most important idea in this lab
> **Q:** When the Writer writes its paragraph, does it "remember" the original question you typed?
> **A:** No. The Writer only ever sees what `run_writer()` was handed — the Analyst's text. This is called **context isolation**: each agent's Claude call only knows what you explicitly pass into it, nothing more. This is exactly the theory concept from Weekend 9, now visible in real code.


### 🚫 Don't mix these up
- ❌ **Wrong:** "The three agents are three separate AI models."
  ✅ **Right:** All three are the **same** Claude model. What makes them "different agents" is only their `system` instructions and what input they receive.
- ❌ **Wrong:** "More agents always means a better answer."
  ✅ **Right:** More agents means more focused steps, but also more API calls (more time, more cost). Use exactly as many as the task needs — three is often enough.
- ❌ **Wrong:** "The pipeline can skip a step if the answer seems obvious."
  ✅ **Right:** The order is fixed: Researcher → Analyst → Writer, every time. Skipping a step is what causes an ungrounded, made-up-sounding final answer.


# Part 4 — Architecture: the whole pipeline in one picture

```
   your question (text)
          |
          v
   +--------------+   5 fact bullets   +-----------+   risks + opportunities   +----------+
   |  Researcher  | -----------------> |  Analyst  | ------------------------> |  Writer  |
   +--------------+                    +-----------+                           +----------+
   system: "facts only"                system: "sort risk/opportunity"        system: "write + verdict"
                                                                                      |
                                                                                      v
                                                                          final recommendation + Verdict
```

**Component responsibilities**
- **Researcher** — turns a question into 5 facts. Failure point: if it invents a "fact," every later step inherits the mistake — this is why its prompt says "facts only."
- **Analyst** — turns facts into risks/opportunities. Failure point: it must be told to use *only* the notes it received, or it may add outside knowledge that sounds plausible but wasn't verified.
- **Writer** — turns analysis into one verdict. Failure point: without "use only what the analysis says," it can drift and sound more confident than the evidence supports.

**Data flow:** every hand-off in this lab is just a Python string being passed as the next `messages=[...]` input. There is no shared memory between agents — only what you explicitly pass along.

**Scale, cost, and reliability**
- **Scale:** 3 agents = 3 API calls per question, run one after another (sequential). More agents or longer chains means more latency.
- **Cost:** you pay for tokens in and out at each step. Haiku is the cheap, fast default for a chain like this.
- **Reliability:** if any one step returns something malformed (e.g. no "Verdict:" line), the whole chain inherits the problem. In production you'd add a check step — that's a natural next lab, not required here.

**Architect's tradeoff:** a fixed 3-step chain is simple, predictable, and cheap to run — perfect for a well-defined task like this one. It is *not* the same thing as letting the model decide how many agents to use and in what order — that's what the next section is about.


# Part 5 — What's next in multi-agent AI (2026)

Your pipeline above has a **fixed** shape: always exactly 3 agents, always in the same order, decided by *you* in advance. That's the simplest and most common design, and it's the right one to learn first.

The newest systems in 2026 take a different approach: instead of you wiring the agents by hand, **the model decides**, on the fly, how many agents to create and what each one should do. The clearest public example of this is Moonshot AI's **Kimi (K2.5 / K2.6) "Agent Swarm."**


## Kimi's Agent Swarm — the model builds its own team

**What is it?**
Agent Swarm is a mode inside Moonshot AI's Kimi models where, instead of following a pipeline you designed, Kimi looks at your task and **decides for itself**: how many sub-agents to create, what each one's job should be, and which ones can run at the same time. Kimi K2.5 (released January 2026) could coordinate up to 100 sub-agents; Kimi K2.6 (released April 2026) raised that to **300 sub-agents** working across up to **4,000 coordinated steps**.

**How is that different from your 3-agent pipeline?**
| Your lab pipeline | Kimi Agent Swarm |
|---|---|
| You choose the roles (Researcher, Analyst, Writer) | The model invents the roles for each task |
| Fixed number of agents (3) | Model decides the number (up to 300) |
| Runs one step at a time, in order (sequential) | Many agents run **at the same time** (parallel) |
| Easy to predict cost and behaviour | Harder to predict — more powerful, but less predictable |

**Why does it run agents in parallel instead of one at a time?**
Think of a research task like "compare pricing, hardware costs, and quantization strategies for deploying open-source models." Your pipeline would do this one step at a time. Kimi's swarm can spin up **three sub-agents at once** — one per topic — so all three finish around the same time instead of one after another. Moonshot reports this parallel approach can cut total run time by roughly **3-4.5x** on wide, research-heavy tasks compared to doing the same work with one agent at a time.

**Why doesn't it always use 300 agents?**
Because splitting work only helps when the work can actually be split. In one public test, Kimi was asked to build a browser game (a "Bubble Shooter" clone) — a task where every part depends on every other part. Kimi's swarm correctly used **just one agent** for that, because there was nothing to parallelize; adding more agents would only add coordination overhead, not speed. This matches a rule worth remembering for your own designs too: **only split a task into parallel agents if the sub-tasks are genuinely independent.**

> **Accuracy note:** Agent Swarm is a feature of Kimi's own product, not a general industry standard, and it runs on a different underlying model from Claude. We cover it here so you can recognise the pattern and discuss it in an interview — the lab itself still uses the fixed 3-agent Claude pipeline, which is the right starting design for most real business tasks.

### Remember this
> A fixed pipeline (like your Researcher → Analyst → Writer) is *you* deciding the org chart in advance. A swarm is the *model* deciding the org chart, per task, in real time. Both are "multi-agent systems" — they just decide differently who does what.


## Other AI agent developments worth knowing (2026)

A few more things happened in the wider agent ecosystem recently that are good to recognise:

- **Multi-agent is becoming the default, not the exception.** Industry surveys in 2026 report that a majority of enterprises already run some form of multi-agent system in production (planner, researcher, executor, and checker-style roles working together), not just single chatbots.
- **Agents are starting to talk to each other, not just to tools.** A protocol called **A2A (Agent-to-Agent)** joined the same open foundation (under the Linux Foundation) as **MCP (Model Context Protocol)** in late 2025. MCP lets an agent use outside tools and data; A2A lets one agent hand off work to a *different* agent, potentially built by a different company. Together they're aimed at letting agents from different vendors cooperate.
- **"Context engineering" is replacing "prompt engineering" as the buzzword.** The bigger design question in 2026 isn't "what do I type into the prompt" — it's "what information does this agent get to see at this step, and what does it not see." That is exactly the hand-off design you practiced in this lab, just given a bigger name at enterprise scale.

### 🚫 Don't mix these up
- ❌ **Wrong:** "Agent Swarm and your Researcher → Analyst → Writer pipeline are basically the same idea."
  ✅ **Right:** Both are "multiple specialised agents," but yours has a **fixed, human-designed** order; a swarm has a **model-decided, dynamic** order. Same family, different design.
- ❌ **Wrong:** "More sub-agents is always faster."
  ✅ **Right:** Parallel agents only save time when the sub-tasks don't depend on each other. Tightly coupled work (like building one coherent app) is often faster with a single agent.

### Interview question
**Q: "Your team's pipeline is a fixed sequence of 3 agents. A newer system like Kimi's Agent Swarm lets the model decide its own agent structure. When would you pick each approach?"**
**Model answer:** "Pick a fixed pipeline like this lab's when the task is well understood and repeatable — you know the steps, and you want predictable cost and behaviour, which matters for anything customer-facing or regulated. Pick a dynamic, swarm-style approach for open-ended tasks where the right breakdown isn't known ahead of time, like broad research or exploration — accepting less predictability in exchange for speed and flexibility on tasks that are genuinely parallelizable."

### ✅ Quick Check
> **Q:** True or False — Kimi's Agent Swarm always uses more agents than a fixed pipeline like the one you built.
> **A:** *False.* It uses as many as the task needs — sometimes just one, if the work can't be split (like the Bubble Shooter game example above).


# Mini Project — "Market-Entry Recommendation Assistant"

**Business use case:** a small strategy team wants fast, structured, first-draft answers to "should we enter market X?" questions, without one person having to research, analyze, and write it all up alone.

**Build it (everything you need is already above):**
1. Reuse your `run_researcher`, `run_analyst`, `run_writer`, and `run_pipeline` functions.
2. Run the pipeline on **three different questions** of your choice (e.g. different cities, different products).
3. Print all three final recommendations side by side.
4. Write 2-3 sentences: which step (Researcher, Analyst, or Writer) would you trust least without a human double-checking it, and why?

**Definition of done**
- The pipeline runs end to end on three different questions without errors.
- Each final answer ends with a clear "Verdict: Yes" or "Verdict: No".
- You can explain, in your own words, what information each agent did and did not see.

**Stretch goal (optional, not required for this lab):** add a fourth function, `run_reviewer`, that receives the research notes, the analysis, *and* the draft, and simply checks whether the draft's verdict matches the analysis's risks. This is the natural next lab — a **Reviewer** agent that checks the other three before anything ships.


## 🛠️ Troubleshooting
| Problem | Likely cause | Fix |
|---|---|---|
| `AuthenticationError` | API key not set or wrong Colab Secret name | Check the Secret is named `MY_API_KEY` and is enabled for this notebook |
| Researcher gives an opinion, not just facts | System prompt too soft | Make the instruction stricter: "facts only, no opinions" |
| Analyst invents a fact not in the notes | Missing "use only what you're given" | Add that sentence to `ANALYST_SYSTEM` |
| Writer has no "Verdict:" line | Output format not enforced clearly enough | Repeat the exact required format in the system prompt |
| Output looks cut off | `max_tokens` too low | Raise `max_tokens` in that agent's `messages.create(...)` call |


---
# Revision Suite


## Session Summary
You built a real 3-agent pipeline — Researcher, Analyst, and Writer — using nothing but three Claude API calls with three different `system` instructions, chained so each agent's output becomes the next agent's input. You saw context isolation in action (each agent only knows what it's handed), you built an architecture picture of the whole system, and you learned how a fixed, human-designed pipeline like yours differs from the newest dynamic multi-agent systems, like Kimi's Agent Swarm, where the model decides its own team structure.


## What You Learned Today — you can now...
- **Explain** what a 3-agent pipeline is in one sentence.
- **Build** a specialised Claude agent using only a `system` instruction and a focused job.
- **Chain** three agents so each one's output becomes the next one's input.
- **Run** a full Researcher → Analyst → Writer pipeline against the live Claude API.
- **Explain** context isolation and why each agent should only see what it needs.
- **Compare** a fixed pipeline to a dynamic agent swarm (Kimi K2.5/K2.6) and say when you'd pick each.


## AI Architect Cheat Sheet

**The one pattern, repeated 3 times:**
```python
reply = client.messages.create(
    model=MODEL, max_tokens=300,
    system="<one job, in plain words>",
    messages=[{"role": "user", "content": "<input from the previous agent>"}],
)
answer = reply.content[0].text
```

**Pipeline in one line:** `writer(analyst(researcher(question)))` — each function's output feeds the next.

**Fixed pipeline vs Agent Swarm**
| | Fixed pipeline (this lab) | Agent Swarm (Kimi K2.5/K2.6) |
|---|---|---|
| Who decides the roles | You | The model |
| Number of agents | Fixed (3 here) | Dynamic (up to 300) |
| Execution | One at a time | Many at once, when possible |
| Best for | Known, repeatable tasks | Open-ended, wide research tasks |

**Model used in this lab:** `claude-haiku-4-5-20251001` — fast and cheap, the right default for a teaching pipeline like this.


## 5-Minute Revision Guide
1. An **agent** here = one Claude API call with its own `system` instruction and its own job.
2. A **pipeline** runs agents in a fixed order: Researcher → Analyst → Writer.
3. A **hand-off** = one agent's text output becomes the next agent's input (just a Python string).
4. **Context isolation**: each agent only ever sees what you explicitly pass it — nothing more.
5. Keep each agent's job **narrow** (facts only / risk-and-opportunity only / recommendation only) so mistakes are easy to trace.
6. A **swarm** (e.g. Kimi Agent Swarm) is the opposite design choice: the model decides the team, the size, and what runs in parallel — powerful for open-ended tasks, less predictable than a fixed pipeline.
7. Only parallelize sub-tasks that are genuinely independent — tightly coupled tasks are often faster with one agent.


## Interview Preparation Notes
- **"What is a multi-agent pipeline?"** Several specialised model calls, each with its own instructions, run in a fixed order, where each one's output feeds the next.
- **"What is context isolation, and why does it matter?"** Each agent only sees what it's explicitly handed, not the full history. It keeps each step focused and makes it easy to trace where an error came from.
- **"Why split Researcher, Analyst, and Writer into three separate calls instead of one big prompt?"** Each instruction stays short and unambiguous, and you can inspect (and fix) any single step without touching the others.
- **"What's the difference between your pipeline and something like Kimi's Agent Swarm?"** Yours is fixed and human-designed; a swarm is dynamic and model-decided, trading predictability for flexibility on open-ended tasks.
- **"When would parallel (swarm-style) agents actually help?"** When sub-tasks are independent of each other — e.g. researching three unrelated topics at once. Tightly coupled tasks (like building one coherent app) don't benefit and can even get slower.


## Assignment
- **Beginner:** run the pipeline on the two example questions already in this notebook and read all three outputs for each.
- **Intermediate:** change the Researcher's instructions to return 3 bullet points instead of 5, and confirm the Analyst and Writer still work correctly with shorter input.
- **Advanced:** write a `run_reviewer` function (see Stretch Goal in the Mini Project) that checks the Writer's draft against the Analyst's risks and prints "MATCH" or "MISMATCH".
- **Project:** complete the Mini Project above on three questions of your own choosing.


## Assessment

**Part A — 10 Multiple Choice**
1. In this lab, what makes the Researcher, Analyst, and Writer different from each other? (a) different AI models (b) different `system` instructions and inputs (c) different programming languages (d) nothing, they're identical
2. A "hand-off" in this lab is: (a) a network call (b) one agent's text output passed as the next agent's input (c) a database write (d) a Colab feature
3. "Context isolation" means: (a) each agent only sees what it's explicitly given (b) all agents share one memory (c) agents can see each other's private thoughts (d) the API key is hidden
4. The order in this lab's pipeline is: (a) Writer → Analyst → Researcher (b) Analyst → Writer → Researcher (c) Researcher → Analyst → Writer (d) any order, it doesn't matter
5. `reply.content[0].text` gives you: (a) the model name (b) the agent's answer text (c) the token count (d) the system prompt
6. The model used by default in this lab is: (a) claude-opus-4-8 (b) claude-haiku-4-5-20251001 (c) gpt-5 (d) kimi-k2.6
7. Kimi's Agent Swarm differs from this lab's pipeline mainly because: (a) it uses a bigger context window only (b) the model decides the number and role of agents dynamically (c) it doesn't use an API (d) it can only do coding tasks
8. In the Bubble Shooter example, Kimi's swarm used just one agent because: (a) it ran out of budget (b) the task was tightly coupled, so splitting it wouldn't help (c) the model was broken (d) games are not supported
9. If the Analyst starts inventing facts not in the research notes, the fix is: (a) use a bigger model (b) tighten the system prompt to say "use only what you're given" (c) delete the Researcher (d) increase max_tokens
10. A2A (Agent-to-Agent) is best described as: (a) a way for one agent to hand off work to another, potentially from a different vendor (b) a new Claude model (c) a pricing plan (d) a Colab feature

**Part B — 5 Short Answer**
1. What is the one thing that changes between your Researcher, Analyst, and Writer functions?
2. Why does the Writer never see the original question directly?
3. What does "context isolation" prevent from happening?
4. Name one situation where a swarm-style (parallel, dynamic) system beats a fixed pipeline, and one where it doesn't.
5. Why should each agent's job be kept narrow instead of broad?

**Part C — 3 Scenarios**
1. Your Writer's final answer contains a number that never appeared in the research notes or the analysis. What's the most likely cause, and how do you fix it?
2. You want to speed up a research task that has three unrelated sub-topics. Would a fixed 3-step pipeline or a swarm-style approach finish faster, and why?
3. A colleague says "let's just use one giant prompt instead of three agents, it's simpler." What's the tradeoff you'd point out?


## Answer Key
**Part A:** 1-b, 2-b, 3-a, 4-c, 5-b, 6-b, 7-b, 8-b, 9-b, 10-a.

**Part B:**
1. Only the `system` instruction and the input text change — it's the same Claude model and the same code pattern every time.
2. Because `run_writer()` is only ever called with the Analyst's output as input — it has no access to anything you didn't explicitly pass it.
3. It prevents an agent from seeing information (or history) it wasn't meant to see, which keeps each step focused and keeps errors traceable to one specific agent.
4. Swarm wins on wide, independent research tasks (e.g. three unrelated topics researched at once); a fixed pipeline wins (or ties) on tightly coupled tasks like building one coherent piece of software, where splitting adds coordination overhead without speed gains.
5. A narrow job is easier to instruct correctly, easier to verify, and easier to debug — if something goes wrong, you know exactly which agent to look at.

**Part C:**
1. The Writer likely drifted beyond its source — tighten `WRITER_SYSTEM` to explicitly say "use only what the analysis says, do not add new facts," and consider adding a Reviewer step (see Stretch Goal) to catch this before it ships.
2. The swarm-style approach would likely finish faster here, because the three sub-topics are independent and can run in parallel; a fixed sequential pipeline would do them one at a time.
3. One giant prompt is simpler to write but harder to instruct precisely (three jobs competing for attention in one prompt) and harder to debug (you can't tell which "part" of the answer went wrong). Three focused agents trade a little extra latency/cost for clarity and easier debugging.


---
### ✅ Lab complete
You built a real, working 3-agent pipeline on the live Claude API, and you can now explain how it compares to the newest dynamic multi-agent systems in the field.

**Official resources**
- Claude Messages API — https://platform.claude.com/docs/en/api/messages
- Claude model IDs and versions — https://platform.claude.com/docs/en/about-claude/models/model-ids-and-versions
- Kimi K2.5 / Agent Swarm overview — https://www.kimi.com/blog/kimi-k2-5.html
- Kimi K2.6 announcement (300-agent swarm) — https://www.marktechpost.com/2026/04/20/moonshot-ai-releases-kimi-k2-6-with-long-horizon-coding-agent-swarm-scaling-to-300-sub-agents-and-4000-coordinated-steps/
